# D1 · 貝葉斯 A/B 測試 — 回答老闆真正的問題

> **核心問題**：A 版轉換率 5.2%、B 版 5.8%。**B 真的比較好嗎？我該不該上線？**

老闆問的是「B 比 A 好的機率是多少」，而 p-value **在定義上無法回答這個問題**。
本 notebook 用 **Beta-Binomial 共軛（封閉解，不需 MCMC）** 走完：
後驗 → P(B>A) → 期望損失決策 → 何時值得上線 → 先驗敏感度。

核心程式在 [`../src/`](../src)，本檔只負責敘事與展示。

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import bayes_ab as ab
from bayes_ab import Prior
from frequentist import two_proportion_z_test
rng = np.random.default_rng(42)

## 1 · 情境與共軛更新

轉換率的共軛更新（計劃書主題二）：$\theta \sim \text{Beta}(1+k,\ 1+n-k)$。
**不需要 MCMC**——這也是它在業界受歡迎的原因（即時計算）。

In [2]:
kA, nA = 208, 4000   # A：5.2%
kB, nB = 232, 4000   # B：5.8%
s = ab.summarize(kA, nA, kB, nB, Prior(1, 1), rng)
print(s.boss_answer())
print('\n後驗 A = Beta%s   後驗 B = Beta%s' % (tuple(round(x,1) for x in s.postA),
                                             tuple(round(x,1) for x in s.postB)))

A 版 5.20%、B 版 5.80%。B 優於 A 的機率 = 88.0%；相對提升的 95% 可信區間 = [-7.0%, +33.7%]（中位數 +11.5%）；若上線 B 但其實 A 較好，期望損失 = 0.0301% 轉換率。

後驗 A = Beta(209, 3793)   後驗 B = Beta(233, 3769)


## 2 · 老闆的問題 vs. p-value

同一份資料，我們用三種方法算 $P(\theta_B>\theta_A)$（互相驗證），並對照頻率派 p-value。

In [3]:
p_grid = s.prob_b_beats_a
p_mc   = ab.prob_b_beats_a_mc(*s.postA, *s.postB, rng)
p_norm = ab.prob_b_beats_a_normal(*s.postA, *s.postB)
z, p_freq = two_proportion_z_test(kA, nA, kB, nB)
print(f'P(B>A)：網格積分={p_grid:.3f} · 蒙地卡羅={p_mc:.3f} · 常態近似={p_norm:.3f}')
print(f'頻率派雙尾 p-value = {p_freq:.3f}  (z={z:.2f})')

P(B>A)：網格積分=0.880 · 蒙地卡羅=0.879 · 常態近似=0.880
頻率派雙尾 p-value = 0.239  (z=1.18)


> **關鍵解讀**：貝葉斯說「B 比 A 好的機率 ≈ 88%」，頻率派卻說「p≈0.24，不顯著」。
> 兩者**不矛盾**——它們回答的是不同問題：
> - p-value：*若 A、B 其實一樣*，看到這麼極端資料的機率。
> - 後驗機率：*給定這份資料*，B 真的比較好的機率。 ← 老闆能拿來做決定的是這個。

![後驗與提升幅度](../figures/01_posteriors_and_lift.png)

左：兩組後驗分佈與 P(B>A)。右：相對提升幅度的後驗，95% 可信區間 **涵蓋 0**——所以「很可能更好，但還不確定」，與 p=0.24 完全一致。

## 3 · 從機率到決策：期望損失

$\mathbb{E}[(\theta_A-\theta_B)^+]$＝「上線 B 但其實 A 較好」的期望後悔（計劃書主題七）。
業界的上線規則就是：**期望損失低於某個閾值 $\varepsilon$ 就上線**。

In [4]:
print(f'上線 B 的期望損失 = {s.expected_loss_choose_b*100:.4f} pp')
print(f'維持 A 的期望損失 = {s.expected_loss_choose_a*100:.4f} pp')
print(f'相對提升 95% 可信區間 = [{s.lift_lo:+.1%}, {s.lift_hi:+.1%}]  中位數 {s.lift_med:+.1%}')

上線 B 的期望損失 = 0.0301 pp
維持 A 的期望損失 = 0.6298 pp
相對提升 95% 可信區間 = [-7.0%, +33.7%]  中位數 +11.5%


## 4 · 何時值得上線（決策成本）

把觀測固定在 5.2% vs 5.8%，看期望損失如何隨資料量縮小；一旦低於容忍度 $\varepsilon$ 就該上線。

![何時上線](../figures/03_when_to_ship.png)

- 左：上線 B 的後悔隨樣本數縮小；降到 $\varepsilon$=0.05pp 以下約需 **~3,000 樣本/組**。而「維持 A」的後悔維持高檔（因為 B 真的較好，不上線的代價持續存在）。
- 右：$n$=4,000/組、$\varepsilon$=0.01pp 時，**值得上線的最小相對提升 ≈ 17%**。

## 5 · 先驗敏感度（誠實的部分）

小樣本時先驗會影響結論。用一個較小的例子（各 200 次）比較三種先驗：

In [5]:
for lab, pr in [('Uniform  Beta(1,1)', Prior(1,1)),
                ('Jeffreys Beta(.5,.5)', Prior(0.5,0.5)),
                ('Informative Beta(5,95)', Prior(5,95))]:
    ss = ab.summarize(10, 200, 17, 200, pr, rng)   # A 5.0% vs B 8.5%
    print(f'{lab:<24} P(B>A)={ss.prob_b_beats_a:.1%}  期望損失(ship B)={ss.expected_loss_choose_b*100:.3g}pp')

Uniform  Beta(1,1)       P(B>A)=91.5%  期望損失(ship B)=0.0998pp
Jeffreys Beta(.5,.5)     P(B>A)=91.9%  期望損失(ship B)=0.0936pp
Informative Beta(5,95)   P(B>A)=88.6%  期望損失(ship B)=0.109pp


![先驗敏感度](../figures/04_prior_sensitivity.png)

一個相當強的先驗（100 個等效觀測、集中在 5% 基準率）只把 P(B>A) 從 91.5% 拉到 88.6%——$n$=200 時資料仍主導，但這個偏移是真的、要報告。**樣本更少時先驗影響更大。**

## 重點

1. p-value 無法回答老闆的問題；貝葉斯後驗機率 + 期望損失可以，而且是老闆聽得懂的話。
2. 決策門檻＝「期望損失 < $\varepsilon$」，$\varepsilon$ 由成本決定——**不是預設的 0.5**。
3. 結論依賴先驗與損失函數的設定 → **貝葉斯不是萬靈丹**，要做敏感度分析並誠實報告。

→ 偷看資料、序貫停止的陷阱見 [`02_peeking_optional_stopping.ipynb`](02_peeking_optional_stopping.ipynb)。